# 07 - Market Odds Benchmark Preparation

This notebook prepares a transparent pre-match market benchmark for Bundesliga evaluation.
The goal is not to evaluate the models yet, but to build a clean and well-documented odds dataset
that can later be compared against the ML and double Poisson models in notebook `08`.

## 1. Imports and Setup

We load the local project paths together with the shared helper functions that download,
clean, normalize, and merge historical pre-match odds.

In [10]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.odds_processing import (
    FOOTBALL_DATA_SOURCE_NAME,
    build_market_benchmark_table,
    build_football_data_bundesliga_url,
    get_available_1x2_odds_columns,
    load_football_data_bundesliga_odds,
    merge_market_odds_with_matches,
    season_id_to_football_data_code,
    summarize_market_merge,
)

## 2. Source Transparency and Notebook Configuration

The odds source used here is [Football-Data.co.uk](https://www.football-data.co.uk/germanym.php).
This source is practical for an academic workflow because it provides open CSV files by season,
together with public notes describing the odds columns and the data collection process.

Important caveats from the source:
- the site distinguishes between regular pre-closing odds and closing odds,
- current fixture odds are collected at fixed times before matchdays,
- since **23 July 2025**, the site explicitly warns that Pinnacle odds have become unreliable,
  and they are no longer included in the calculation of market average and maximum odds.

For that reason, the notebook keeps both opening and closing odds when available,
but uses a transparent benchmark priority with market-average closing odds first.

In [11]:
SEASON_IDS = [2023, 2024, 2025]
RAW_ODDS_DIR = RAW_DATA_DIR / 'football_data'
PROCESSED_MARKET_PATH = PROCESSED_DATA_DIR / 'market_odds_bundesliga.csv'
PROCESSED_MATCH_BENCHMARK_PATH = PROCESSED_DATA_DIR / 'market_benchmark_matches.csv'
SOURCE_METADATA_PATH = PROCESSED_DATA_DIR / 'market_odds_source_metadata.csv'

source_metadata = pd.DataFrame(
    [
        {
            'source_name': FOOTBALL_DATA_SOURCE_NAME,
            'country_page_url': 'https://www.football-data.co.uk/germanym.php',
            'notes_url': 'https://www.football-data.co.uk/notes.txt',
            'general_data_page_url': 'https://www.football-data.co.uk/data.php',
            'coverage': 'Bundesliga historical CSV files by season',
            'benchmark_priority': 'market-average closing -> bet365 closing -> market-average opening -> bet365 opening',
            'double_chance_policy': 'Derived from normalized 1X2 implied probabilities',
            'pinnacle_caveat': 'Football-Data warns from 2025-07-23 that Pinnacle odds are unreliable and excluded from market averages',
        }
    ]
)

download_targets = pd.DataFrame(
    {
        'season_id': SEASON_IDS,
        'football_data_code': [season_id_to_football_data_code(season_id) for season_id in SEASON_IDS],
        'source_url': [build_football_data_bundesliga_url(season_id) for season_id in SEASON_IDS],
    }
)

display(source_metadata)
display(download_targets)

,source_name,country_page_url,notes_url,general_data_page_url,coverage,benchmark_priority,double_chance_policy,pinnacle_caveat
0,Football-Data.co.uk,https://www.football-data.co.uk/germanym.php,https://www.football-data.co.uk/notes.txt,https://www.football-data.co.uk/data.php,Bundesliga historical CSV files by season,market-average closing -> bet365 closing -> ma...,Derived from normalized 1X2 implied probabilities,Football-Data warns from 2025-07-23 that Pinna...


,season_id,football_data_code,source_url
0,2023,2324,https://www.football-data.co.uk/mmz4281/2324/D...
1,2024,2425,https://www.football-data.co.uk/mmz4281/2425/D...
2,2025,2526,https://www.football-data.co.uk/mmz4281/2526/D...


## 3. Load the Local Match Table

We load the local advanced match table because the final output of this notebook should be
aligned with the same match identifiers, targets, and team names used by the prediction notebooks.

In [12]:
features_path = PROCESSED_DATA_DIR / 'match_features_advanced.parquet'
if not features_path.exists():
    features_path = PROCESSED_DATA_DIR / 'match_features_advanced.csv'

try:
    df_matches = pd.read_parquet(features_path) if features_path.suffix == '.parquet' else pd.read_csv(features_path)
except Exception:
    fallback_csv_path = PROCESSED_DATA_DIR / 'match_features_advanced.csv'
    df_matches = pd.read_csv(fallback_csv_path)

df_matches['date'] = pd.to_datetime(df_matches['date'], errors='coerce')

match_core_cols = [
    col for col in [
        'game_id', 'date', 'season_id', 'home_team', 'away_team',
        'home_goals', 'away_goals', 'target_1x2', 'home_win', 'draw', 'away_win', 'matchday'
    ]
    if col in df_matches.columns
]

print('Local match table shape:', df_matches.shape)
display(df_matches[match_core_cols].head())
print('Seasons available in local match table:', sorted(df_matches['season_id'].dropna().unique().tolist()))


Local match table shape: (882, 249)


,game_id,date,season_id,home_team,away_team,home_goals,away_goals,target_1x2,home_win,draw,away_win
0,23065,2023-08-18 18:30:00,2023,Werder Bremen,Bayern Munich,0,4,A,0,0,1
1,23066,2023-08-19 13:30:00,2023,Bayer Leverkusen,RasenBallsport Leipzig,3,2,H,1,0,0
2,23067,2023-08-19 13:30:00,2023,Wolfsburg,FC Heidenheim,2,0,H,1,0,0
3,23068,2023-08-19 13:30:00,2023,Hoffenheim,Freiburg,1,2,A,0,0,1
4,23069,2023-08-19 13:30:00,2023,Augsburg,Borussia M.Gladbach,4,4,D,0,1,0


Seasons available in local match table: [2023, 2024, 2025]


## 4. Download Raw Bundesliga Odds

We download one raw CSV per season directly from Football-Data and also save a local copy
under `data/raw/football_data`, so the acquisition step stays reproducible.

In [13]:
RAW_ODDS_DIR.mkdir(parents=True, exist_ok=True)

raw_odds = load_football_data_bundesliga_odds(
    season_ids=SEASON_IDS,
    raw_save_dir=RAW_ODDS_DIR,
)

combined_raw_path = RAW_ODDS_DIR / 'bundesliga_combined_raw.csv'
raw_odds.to_csv(combined_raw_path, index=False)

print('Raw odds shape:', raw_odds.shape)
display(raw_odds[['season_id', 'Date', 'HomeTeam', 'AwayTeam']].head())
print('Raw files saved to:', RAW_ODDS_DIR)

Raw odds shape: (882, 164)


,season_id,Date,HomeTeam,AwayTeam
0,2023,18/08/2023,Werder Bremen,Bayern Munich
1,2023,19/08/2023,Augsburg,M'gladbach
2,2023,19/08/2023,Hoffenheim,Freiburg
3,2023,19/08/2023,Leverkusen,RB Leipzig
4,2023,19/08/2023,Stuttgart,Bochum


Raw files saved to: C:\Users\cerve\Desktop\DP\match_prediction\data\raw\football_data


## 5. Inspect Available Odds Fields

Football-Data files may contain several bookmaker-specific odds columns as well as market-average columns.
This section shows which common 1X2 triplets are actually available in the downloaded data.

In [14]:
available_1x2_columns = get_available_1x2_odds_columns(raw_odds)
display(available_1x2_columns)

example_odds_cols = [
    col for col in [
        'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam',
        'AvgH', 'AvgD', 'AvgA',
        'AvgCH', 'AvgCD', 'AvgCA',
        'B365H', 'B365D', 'B365A',
        'B365CH', 'B365CD', 'B365CA',
        'PSH', 'PSD', 'PSA',
        'PSCH', 'PSCD', 'PSCA',
    ]
    if col in raw_odds.columns
]
display(raw_odds[example_odds_cols].head())

,odds_source,home_col,draw_col,away_col,complete_triplet_available,n_complete_rows
0,market_average_open,AvgH,AvgD,AvgA,True,882
1,bet365_open,B365H,B365D,B365A,True,882
2,pinnacle_open,PSH,PSD,PSA,True,762
3,market_average_close,AvgCH,AvgCD,AvgCA,True,882
4,bet365_close,B365CH,B365CD,B365CA,True,882
5,pinnacle_close,PSCH,PSCD,PSCA,True,761


,Div,Date,Time,HomeTeam,AwayTeam,AvgH,AvgD,AvgA,AvgCH,AvgCD,...,B365A,B365CH,B365CD,B365CA,PSH,PSD,PSA,PSCH,PSCD,PSCA
0,D1,18/08/2023,19:30,Werder Bremen,Bayern Munich,8.50,6.09,1.32,8.69,6.05,...,1.30,8.50,6.0,1.30,8.59,6.36,1.33,8.80,6.30,1.31
1,D1,19/08/2023,14:30,Augsburg,M'gladbach,2.72,3.69,2.47,2.79,3.67,...,2.45,2.80,3.6,2.38,2.74,3.78,2.51,2.92,3.72,2.43
2,D1,19/08/2023,14:30,Hoffenheim,Freiburg,2.35,3.63,2.94,2.20,3.68,...,2.90,2.15,3.6,3.10,2.43,3.67,2.92,2.20,3.74,3.32
3,D1,19/08/2023,14:30,Leverkusen,RB Leipzig,2.47,3.62,2.78,2.44,3.60,...,2.75,2.38,3.6,2.80,2.50,3.61,2.85,2.48,3.63,2.98
4,D1,19/08/2023,14:30,Stuttgart,Bochum,1.69,4.17,4.69,1.76,4.02,...,4.50,1.73,4.2,4.20,1.70,4.25,4.78,1.78,4.16,4.52


## 6. Build the Market Benchmark Table

The helper pipeline does the following:
- standardizes match dates and team names,
- resolves opening odds and closing odds separately,
- builds one benchmark triplet using the priority defined above,
- converts decimal odds into implied probabilities,
- normalizes the probabilities to remove the overround,
- derives double-chance benchmarks such as `1X` and `X2`.

This is important because the project also contains binary models,
and the `X2` benchmark is the natural market analogue of the "away not lose" target.

In [15]:
market_odds = build_market_benchmark_table(raw_odds)

benchmark_summary = (
    market_odds.groupby(['season_id', 'benchmark_stage', 'benchmark_odds_source'], dropna=False)
    .size()
    .rename('n_matches')
    .reset_index()
    .sort_values(['season_id', 'n_matches'], ascending=[True, False])
)

display(benchmark_summary)
display(
    market_odds[
        [
            'season_id', 'Date', 'HomeTeam', 'AwayTeam',
            'opening_odds_source', 'closing_odds_source',
            'benchmark_odds_source', 'benchmark_stage',
            'benchmark_home_odds', 'benchmark_draw_odds', 'benchmark_away_odds',
            'benchmark_overround',
            'benchmark_home_prob', 'benchmark_draw_prob', 'benchmark_away_prob',
            'benchmark_home_not_lose_prob', 'benchmark_away_not_lose_prob', 'benchmark_no_draw_prob',
        ]
    ].head(10)
)

,season_id,benchmark_stage,benchmark_odds_source,n_matches
0,2023,closing,market_average_close,306
1,2024,closing,market_average_close,306
2,2025,closing,market_average_close,270


,season_id,Date,HomeTeam,AwayTeam,opening_odds_source,closing_odds_source,benchmark_odds_source,benchmark_stage,benchmark_home_odds,benchmark_draw_odds,benchmark_away_odds,benchmark_overround,benchmark_home_prob,benchmark_draw_prob,benchmark_away_prob,benchmark_home_not_lose_prob,benchmark_away_not_lose_prob,benchmark_no_draw_prob
0,2023,2023-08-18,Werder Bremen,Bayern Munich,market_average_open,market_average_close,market_average_close,closing,8.69,6.05,1.31,1.043723,0.110254,0.158365,0.731381,0.268619,0.889746,0.841635
1,2023,2023-08-19,Augsburg,M'gladbach,market_average_open,market_average_close,market_average_close,closing,2.79,3.67,2.42,1.044126,0.343276,0.260964,0.395760,0.604240,0.656724,0.739036
2,2023,2023-08-19,Hoffenheim,Freiburg,market_average_open,market_average_close,market_average_close,closing,2.20,3.68,3.15,1.043745,0.435495,0.260350,0.304155,0.695845,0.564505,0.739650
3,2023,2023-08-19,Leverkusen,RB Leipzig,market_average_open,market_average_close,market_average_close,closing,2.44,3.60,2.81,1.043486,0.392757,0.266202,0.341041,0.658959,0.607243,0.733798
4,2023,2023-08-19,Stuttgart,Bochum,market_average_open,market_average_close,market_average_close,closing,1.76,4.02,4.32,1.048420,0.541941,0.237268,0.220791,0.779209,0.458059,0.762732
5,2023,2023-08-19,Wolfsburg,Heidenheim,market_average_open,market_average_close,market_average_close,closing,1.64,4.28,5.03,1.042208,0.585062,0.224183,0.190756,0.809244,0.414938,0.775817
6,2023,2023-08-19,Dortmund,FC Koln,market_average_open,market_average_close,market_average_close,closing,1.46,4.95,6.33,1.044930,0.655481,0.193334,0.151185,0.848815,0.344519,0.806666
7,2023,2023-08-20,Union Berlin,Mainz,market_average_open,market_average_close,market_average_close,closing,2.15,3.32,3.61,1.043329,0.445800,0.288696,0.265504,0.734496,0.554200,0.711304
8,2023,2023-08-20,Ein Frankfurt,Darmstadt,market_average_open,market_average_close,market_average_close,closing,1.48,4.69,6.50,1.042741,0.647980,0.204480,0.147540,0.852460,0.352020,0.795520
9,2023,2023-08-25,RB Leipzig,Stuttgart,market_average_open,market_average_close,market_average_close,closing,1.58,4.43,5.40,1.043830,0.606336,0.216255,0.177409,0.822591,0.393664,0.783745


## 7. Merge the Market Benchmark onto the Local Match Dataset

The merge is done on season, match date, and normalized home/away team names.
This section is purely diagnostic: before using the odds benchmark in notebook `08`,
we want to know exactly how much of the local match table is covered.

In [16]:
matches_with_market = merge_market_odds_with_matches(
    df_matches=df_matches,
    df_market_odds=market_odds,
)

merge_summary = summarize_market_merge(matches_with_market)
display(merge_summary)

unmatched_matches = matches_with_market.loc[
    matches_with_market['benchmark_home_odds'].isna(),
    [col for col in ['season_id', 'date', 'home_team', 'away_team'] if col in matches_with_market.columns]
].drop_duplicates()

print('Sample of unmatched matches:')
display(unmatched_matches.head(15))

,season_id,n_matches,n_with_market_odds,merge_coverage
0,2023,306,272,0.888889
1,2024,306,271,0.885621
2,2025,270,240,0.888889


Sample of unmatched matches:


,season_id,date,home_team,away_team
6,2023,2023-08-19 16:30:00,Borussia Dortmund,FC Cologne
12,2023,2023-08-26 13:30:00,Bochum,Borussia Dortmund
18,2023,2023-09-01 18:30:00,Borussia Dortmund,FC Heidenheim
29,2023,2023-09-16 13:30:00,Freiburg,Borussia Dortmund
40,2023,2023-09-23 13:30:00,Borussia Dortmund,Wolfsburg
45,2023,2023-09-29 18:30:00,Hoffenheim,Borussia Dortmund
55,2023,2023-10-07 13:30:00,Borussia Dortmund,Union Berlin
63,2023,2023-10-20 18:30:00,Borussia Dortmund,Werder Bremen
79,2023,2023-10-29 14:30:00,Eintracht Frankfurt,Borussia Dortmund
87,2023,2023-11-04 17:30:00,Borussia Dortmund,Bayern Munich


## 8. Inspect the Final Benchmark Probabilities

At this point we already have the key market benchmark quantities needed for later evaluation:
- normalized `1X2` probabilities,
- the market-implied binary benchmark `P(home_win)`,
- the market-implied double-chance benchmark `P(away_not_lose) = P(draw) + P(away_win)`.

In [17]:
probability_cols = [
    col for col in [
        'benchmark_home_prob', 'benchmark_draw_prob', 'benchmark_away_prob',
        'benchmark_home_not_lose_prob', 'benchmark_away_not_lose_prob', 'benchmark_no_draw_prob',
        'benchmark_home_not_lose_fair_odds', 'benchmark_away_not_lose_fair_odds', 'benchmark_no_draw_fair_odds',
    ]
    if col in matches_with_market.columns
]

display(matches_with_market[probability_cols].describe().T)

current_season_market = matches_with_market[matches_with_market['season_id'] == 2025].copy()
print('Current-season benchmark availability:', current_season_market['benchmark_home_odds'].notna().mean())
display(
    current_season_market[
        [
            col for col in [
                'date', 'home_team', 'away_team', 'target_1x2', 'home_win',
                'benchmark_home_prob', 'benchmark_draw_prob', 'benchmark_away_prob',
                'benchmark_away_not_lose_prob', 'benchmark_pred_1x2', 'benchmark_pred_home_win_binary'
            ]
            if col in current_season_market.columns
        ]
    ].head(15)
)

,count,mean,std,min,25%,50%,75%,max
benchmark_home_prob,783.0,0.450337,0.180431,0.059680,0.342470,0.438287,0.553184,0.924823
benchmark_draw_prob,783.0,0.235130,0.050513,0.053198,0.212215,0.252549,0.271293,0.323230
benchmark_away_prob,783.0,0.314533,0.163923,0.021979,0.207729,0.296498,0.386646,0.838131
benchmark_home_not_lose_prob,783.0,0.685467,0.163923,0.161869,0.613354,0.703502,0.792271,0.978021
benchmark_away_not_lose_prob,783.0,0.549663,0.180431,0.075177,0.446816,0.561713,0.657530,0.940320
benchmark_no_draw_prob,783.0,0.764870,0.050513,0.676770,0.728707,0.747451,0.787785,0.946802
benchmark_home_not_lose_fair_odds,783.0,1.597901,0.651219,1.022473,1.262195,1.421461,1.630382,6.177843
benchmark_away_not_lose_fair_odds,783.0,2.173047,1.305478,1.063467,1.520844,1.780269,2.238059,13.301921
benchmark_no_draw_fair_odds,783.0,1.312720,0.080696,1.056187,1.269382,1.337880,1.372294,1.477608


Current-season benchmark availability: 0.8888888888888888


,date,home_team,away_team,target_1x2,home_win,benchmark_home_prob,benchmark_draw_prob,benchmark_away_prob,benchmark_away_not_lose_prob,benchmark_pred_1x2,benchmark_pred_home_win_binary
612,2025-08-22 18:30:00,Bayern Munich,RasenBallsport Leipzig,H,1,0.768735,0.135465,0.095800,0.231265,H,1
613,2025-08-23 13:30:00,FC Heidenheim,Wolfsburg,A,0,0.344039,0.278895,0.377066,0.655961,A,0
614,2025-08-23 13:30:00,Bayer Leverkusen,Hoffenheim,A,0,0.550858,0.226980,0.222162,0.449142,H,1
615,2025-08-23 13:30:00,Union Berlin,VfB Stuttgart,H,1,0.241086,0.249377,0.509538,0.758914,A,0
616,2025-08-23 13:30:00,Eintracht Frankfurt,Werder Bremen,H,1,0.614992,0.206800,0.178208,0.385008,H,1
617,2025-08-23 13:30:00,Freiburg,Augsburg,A,0,0.488078,0.269140,0.242781,0.511922,H,0
618,2025-08-23 16:30:00,St. Pauli,Borussia Dortmund,D,0,NaN,NaN,NaN,NaN,NaN,NaN
619,2025-08-24 13:30:00,Mainz 05,FC Cologne,A,0,0.483397,0.261840,0.254763,0.516603,H,0
620,2025-08-24 15:30:00,Borussia M.Gladbach,Hamburger SV,D,0,0.498460,0.243434,0.258107,0.501540,H,0
621,2025-08-29 18:30:00,Hamburger SV,St. Pauli,A,0,0.389656,0.292847,0.317497,0.610344,H,0


## 9. Save the Outputs for Notebook 08

We save three artifacts:
- a source metadata table for transparency,
- a processed market-odds table at the odds-source level,
- a match-level benchmark table already merged onto the local match dataset.

Notebook `08` can then focus only on evaluation and comparison, without repeating the acquisition logic.

In [18]:
source_metadata.to_csv(SOURCE_METADATA_PATH, index=False)
market_odds.to_csv(PROCESSED_MARKET_PATH, index=False)
matches_with_market.to_csv(PROCESSED_MATCH_BENCHMARK_PATH, index=False)

print('Saved:')
print('-', SOURCE_METADATA_PATH)
print('-', PROCESSED_MARKET_PATH)
print('-', PROCESSED_MATCH_BENCHMARK_PATH)

Saved:
- C:\Users\cerve\Desktop\DP\match_prediction\data\processed\market_odds_source_metadata.csv
- C:\Users\cerve\Desktop\DP\match_prediction\data\processed\market_odds_bundesliga.csv
- C:\Users\cerve\Desktop\DP\match_prediction\data\processed\market_benchmark_matches.csv
